# Model Comparison

Two-stage comparison from the saved test-set results (`04_test_set_evaluation.ipynb` output): (1) single-task vs multi-task, per task, per loss-weighting strategy; (2) EW vs UW vs DWA head-to-head. With only 3 seeds per model, a difference is only called meaningful if it exceeds the larger of the two configurations' seed-to-seed standard deviations -- otherwise the configurations are treated as equivalent, which is itself a valid finding. Pure analysis on saved CSVs; no GPU or checkpoints needed.

In [ ]:
!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'

In [ ]:
import pandas as pd
from comparison import compare_stl_vs_mtl, compare_efficiency, compare_strategies

single_task_df = pd.read_csv(f'{RESULTS_DIR}/single_task_test_results.csv')
multitask_df = pd.read_csv(f'{RESULTS_DIR}/multitask_test_results.csv')
single_task_df.head(), multitask_df.head()

## Stage 1a -- Species: Model A vs each MTL strategy's species head

In [ ]:
species_stl = single_task_df[single_task_df['model'] == 'ModelA_species']['f1_macro']
species_comparison = compare_stl_vs_mtl(
    species_stl.mean(), species_stl.std(), multitask_df, 'species_f1_macro'
)
species_comparison

## Stage 1b -- Freshness: Model B vs each MTL strategy's freshness head

In [ ]:
freshness_stl = single_task_df[single_task_df['model'] == 'ModelB_freshness']['f1_macro']
freshness_comparison = compare_stl_vs_mtl(
    freshness_stl.mean(), freshness_stl.std(), multitask_df, 'freshness_f1_macro'
)
freshness_comparison

## Stage 1c -- Efficiency: two separate STL models vs one shared-backbone MTL model

STL total is Model A + Model B combined, since getting both predictions under the single-task approach requires running both models.

In [ ]:
stl_total_params = (
    single_task_df[single_task_df['model'] == 'ModelA_species']['params'].mean()
    + single_task_df[single_task_df['model'] == 'ModelB_freshness']['params'].mean()
)
stl_total_inference_ms = (
    single_task_df[single_task_df['model'] == 'ModelA_species']['inference_ms'].mean()
    + single_task_df[single_task_df['model'] == 'ModelB_freshness']['inference_ms'].mean()
)

params_savings = compare_efficiency(stl_total_params, multitask_df, 'params')
time_savings = compare_efficiency(stl_total_inference_ms, multitask_df, 'inference_ms')
params_savings, time_savings

## Stage 2 -- EW vs UW vs DWA head-to-head

In [ ]:
strategy_ranking = pd.DataFrame([
    compare_strategies(multitask_df, 'species_f1_macro'),
    compare_strategies(multitask_df, 'freshness_f1_macro'),
    compare_strategies(multitask_df, 'joint_accuracy'),
])
strategy_ranking

If `exceeds_seed_variation` is False for a metric, the three strategies are equivalent on that metric within the range of random-seed variation -- report this directly rather than picking a nominal winner.

In [ ]:
species_comparison.to_csv(f'{RESULTS_DIR}/comparison_stl_vs_mtl_species.csv', index=False)
freshness_comparison.to_csv(f'{RESULTS_DIR}/comparison_stl_vs_mtl_freshness.csv', index=False)
params_savings.to_csv(f'{RESULTS_DIR}/comparison_efficiency_params.csv', index=False)
time_savings.to_csv(f'{RESULTS_DIR}/comparison_efficiency_time.csv', index=False)
strategy_ranking.to_csv(f'{RESULTS_DIR}/comparison_strategy_ranking.csv', index=False)
print('saved to', RESULTS_DIR)

Saved to `RESULTS_DIR` on Drive. To add these to the GitHub repository, download the CSVs and commit them from a machine with push access.